In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import json, os, warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score,
                             normalized_mutual_info_score)
from scipy.cluster.hierarchy import dendrogram, linkage
import seaborn as sns

PLOT_DIR = r'c:\ML\EXP9\plots'
os.makedirs(PLOT_DIR, exist_ok=True)

In [ ]:
# ── 1. Load Dataset ────────────────────────────────────────────────────────────
print("Loading HAR dataset...")
har = fetch_openml('har', version=1, as_frame=True, parser='auto')
X_full = har.data.values.astype(float)
le = LabelEncoder()
y_true = le.fit_transform(har.target.astype(str))
activity_names = {
    '1': 'WALKING', '2': 'WALKING_UPSTAIRS', '3': 'WALKING_DOWNSTAIRS',
    '4': 'SITTING', '5': 'STANDING', '6': 'LAYING'
}
print(f"Full dataset: {X_full.shape[0]} samples, {X_full.shape[1]} features")
print(f"Activities: {le.classes_}")

In [ ]:
# ── 2. Subsample for speed (keep 3000 samples, stratified) ────────────────────
np.random.seed(42)
idx = []
for c in np.unique(y_true):
    c_idx = np.where(y_true == c)[0]
    idx.extend(np.random.choice(c_idx, size=min(500, len(c_idx)), replace=False))
idx = np.array(idx)
X_sub = X_full[idx]
y_sub = y_true[idx]
print(f"Subsampled: {X_sub.shape[0]} samples")

In [ ]:
# ── 3. Preprocess ──────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sub)

# PCA to 50 components for clustering, 2D for visualization
pca50 = PCA(n_components=50, random_state=42)
X_pca50 = pca50.fit_transform(X_scaled)
pca2d = PCA(n_components=2, random_state=42)
X_2d = pca2d.fit_transform(X_scaled)
print(f"PCA 50-component variance explained: {pca50.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# ── 4. Elbow + Silhouette for K-Means ─────────────────────────────────────────
print("Running K-Means elbow analysis...")
ks = [2, 3, 4, 5, 6, 7, 8]
wcss_list, sil_list = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca50)
    wcss_list.append(km.inertia_)
    sil_list.append(silhouette_score(X_pca50, labels, sample_size=1000, random_state=42))
    print(f"  k={k}: WCSS={km.inertia_:.1f}, Silhouette={sil_list[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(ks, wcss_list, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(6, color='red', linestyle='--', label='k=6 (true classes)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_title('Elbow Method'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(ks, sil_list, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(6, color='red', linestyle='--', label='k=6')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette vs k'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'kmeans_elbow_silhouette.png'))
plt.savefig(os.path.join(PLOT_DIR, 'kmeans_elbow_silhouette.eps'), format='eps')
plt.close()
print("Saved elbow/silhouette plot.")

In [ ]:
# ── 5. K-Means with k=6 ───────────────────────────────────────────────────────
print("Fitting final KMeans k=6...")
km6 = KMeans(n_clusters=6, random_state=42, n_init=20)
km_labels = km6.fit_predict(X_pca50)

km_sil = silhouette_score(X_pca50, km_labels, sample_size=1000, random_state=42)
km_db  = davies_bouldin_score(X_pca50, km_labels)
km_ch  = calinski_harabasz_score(X_pca50, km_labels)
km_ari = adjusted_rand_score(y_sub, km_labels)
km_nmi = normalized_mutual_info_score(y_sub, km_labels)
print(f"K-Means: Sil={km_sil:.4f} DB={km_db:.4f} CH={km_ch:.1f} ARI={km_ari:.4f} NMI={km_nmi:.4f}")

# Scatter plot
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_2d[:,0], X_2d[:,1], c=km_labels, cmap='tab10', s=10, alpha=0.7)
ax.set_title('K-Means (k=6) Clusters -- PCA 2D')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'kmeans_clusters.png'))
plt.savefig(os.path.join(PLOT_DIR, 'kmeans_clusters.eps'), format='eps')
plt.close()

In [ ]:
# ── 6. DBSCAN ─────────────────────────────────────────────────────────────────
print("Running DBSCAN...")
db = DBSCAN(eps=5.5, min_samples=15, n_jobs=1)
db_labels = db.fit_predict(X_pca50)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise_db    = list(db_labels).count(-1)
print(f"DBSCAN: {n_clusters_db} clusters, {n_noise_db} noise points")

mask = db_labels != -1
if mask.sum() > 1 and len(set(db_labels[mask])) > 1:
    db_sil = silhouette_score(X_pca50[mask], db_labels[mask], sample_size=min(1000, mask.sum()), random_state=42)
    db_ari = adjusted_rand_score(y_sub[mask], db_labels[mask])
    db_nmi = normalized_mutual_info_score(y_sub[mask], db_labels[mask])
else:
    db_sil = db_ari = db_nmi = float('nan')
print(f"DBSCAN: Sil={db_sil:.4f} ARI={db_ari:.4f} NMI={db_nmi:.4f}")

fig, ax = plt.subplots(figsize=(9, 7))
colors = np.where(db_labels == -1, 'lightgrey', 'steelblue')
unique_labels = sorted(set(db_labels))
cmap = plt.get_cmap('tab10')
for label in unique_labels:
    mask_l = db_labels == label
    color = 'lightgrey' if label == -1 else cmap(label % 10)
    lname = 'Noise' if label == -1 else f'Cluster {label}'
    ax.scatter(X_2d[mask_l,0], X_2d[mask_l,1], c=[color]*mask_l.sum(), s=6, alpha=0.6, label=lname)
ax.set_title(f'DBSCAN Clusters ({n_clusters_db} clusters, {n_noise_db} noise) -- PCA 2D')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
if len(unique_labels) <= 12:
    ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'dbscan_clusters.png'))
plt.savefig(os.path.join(PLOT_DIR, 'dbscan_clusters.eps'), format='eps')
plt.close()

In [ ]:
# ── 7. Hierarchical (Agglomerative, Ward) ─────────────────────────────────────
print("Running Agglomerative Clustering (Ward, k=6)...")
hac = AgglomerativeClustering(n_clusters=6, linkage='ward')
hac_labels = hac.fit_predict(X_pca50)

hac_sil = silhouette_score(X_pca50, hac_labels, sample_size=1000, random_state=42)
hac_db  = davies_bouldin_score(X_pca50, hac_labels)
hac_ch  = calinski_harabasz_score(X_pca50, hac_labels)
hac_ari = adjusted_rand_score(y_sub, hac_labels)
hac_nmi = normalized_mutual_info_score(y_sub, hac_labels)
print(f"HAC: Sil={hac_sil:.4f} DB={hac_db:.4f} CH={hac_ch:.1f} ARI={hac_ari:.4f} NMI={hac_nmi:.4f}")

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_2d[:,0], X_2d[:,1], c=hac_labels, cmap='tab10', s=10, alpha=0.7)
ax.set_title('Hierarchical (Ward, k=6) -- PCA 2D')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'hac_clusters.png'))
plt.savefig(os.path.join(PLOT_DIR, 'hac_clusters.eps'), format='eps')
plt.close()

# Dendrogram (on a small subsample)
print("Computing dendrogram (subsample 300)...")
idx300 = np.random.choice(len(X_pca50), size=300, replace=False)
Z = linkage(X_pca50[idx300], method='ward')
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=8)
ax.set_title('Dendrogram (Ward Linkage, 300 samples, truncated)')
ax.set_xlabel('Sample index'); ax.set_ylabel('Distance')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'dendrogram.png'))
plt.savefig(os.path.join(PLOT_DIR, 'dendrogram.eps'), format='eps')
plt.close()

In [ ]:
# ── 8. True Labels 2D Plot ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
act_labels = {0:'WALKING',1:'WALKING_UP',2:'WALKING_DN',3:'SITTING',4:'STANDING',5:'LAYING'}
cmap10 = plt.get_cmap('tab10')
for i, name in act_labels.items():
    mask_a = y_sub == i
    ax.scatter(X_2d[mask_a,0], X_2d[mask_a,1], c=[cmap10(i)]*mask_a.sum(),
               s=8, alpha=0.7, label=name)
ax.set_title('True Activity Labels -- PCA 2D'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'true_labels_pca.png'))
plt.savefig(os.path.join(PLOT_DIR, 'true_labels_pca.eps'), format='eps')
plt.close()

In [ ]:
# ── 9. Metric Comparison Bar Plot ─────────────────────────────────────────────
metrics_names = ['Silhouette', 'Davies-Bouldin', 'ARI', 'NMI']
km_vals  = [km_sil,  km_db,   km_ari,  km_nmi]
db_vals  = [db_sil,  float('nan'), db_ari, db_nmi]
hac_vals = [hac_sil, hac_db, hac_ari, hac_nmi]

x = np.arange(len(metrics_names))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x-0.25, km_vals,  0.25, label='K-Means', color='#3498db')
ax.bar(x,      db_vals,  0.25, label='DBSCAN',  color='#e74c3c')
ax.bar(x+0.25, hac_vals, 0.25, label='HAC',     color='#2ecc71')
ax.set_xticks(x); ax.set_xticklabels(metrics_names)
ax.set_title('Clustering Metrics Comparison'); ax.legend(); ax.grid(axis='y')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'metrics_comparison.png'))
plt.savefig(os.path.join(PLOT_DIR, 'metrics_comparison.eps'), format='eps')
plt.close()
print("All plots saved.")

In [ ]:
# ── 10. Save Results ───────────────────────────────────────────────────────────
results = {
    'dataset': {'n_samples': int(X_sub.shape[0]), 'n_features': int(X_full.shape[1]),
                'n_classes': 6},
    'pca50_variance_pct': round(float(pca50.explained_variance_ratio_.sum()*100), 1),
    'elbow': {'k': ks, 'wcss': [round(w,1) for w in wcss_list],
              'silhouette': [round(s,4) for s in sil_list]},
    'kmeans': {'k': 6, 'silhouette': round(km_sil,4), 'davies_bouldin': round(km_db,4),
               'calinski_harabasz': round(km_ch,1), 'ari': round(km_ari,4), 'nmi': round(km_nmi,4)},
    'dbscan': {'eps': 5.5, 'min_samples': 15, 'n_clusters': n_clusters_db, 'n_noise': n_noise_db,
               'silhouette': round(float(db_sil) if db_sil==db_sil else 0,4),
               'ari': round(float(db_ari) if db_ari==db_ari else 0,4),
               'nmi': round(float(db_nmi) if db_nmi==db_nmi else 0,4)},
    'hac': {'n_clusters': 6, 'linkage': 'ward',
            'silhouette': round(hac_sil,4), 'davies_bouldin': round(hac_db,4),
            'calinski_harabasz': round(hac_ch,1), 'ari': round(hac_ari,4), 'nmi': round(hac_nmi,4)},
}
with open(r'c:\ML\EXP9\results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved to results.json")